## Authors of code:
- Viktor Kolak

In [ ]:
import os 
import pandas as pd 
import numpy as np
import tensorflow as tf
import keras 
from keras import layers, models, Input
from keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:

df_data = pd.read_csv(filepath_or_buffer=REPLACE!!)

In [26]:
x_data = df_data.drop(columns=['key','quality'])
y_key_data = df_data['key']
y_quality_data = df_data['quality']


#split into train validate and test
x_train, x_test, y_key_train, y_key_test, y_quality_train, y_quality_test = train_test_split(x_data, 
    y_key_data, 
    y_quality_data, 
    test_size=0.2, 
    random_state=42)

print(x_test)
print(x_train)
x_test.to_numpy()
x_train.to_numpy()
y_key_test.to_numpy()
y_key_train.to_numpy()
y_quality_test.to_numpy()
y_quality_train.to_numpy()


#add padding 
padding = 0 


#make it into the correct shape

          timestamp  chroma_0  chroma_1  chroma_2  chroma_3  chroma_4  \
409405   108.065669  0.000000  0.034222  2.364240  0.007879  0.000000   
383496    26.284989  0.006861  0.196629  0.384269  0.000000  0.000000   
541458   185.666757  2.286660  0.126700  0.000000  0.000000  0.000000   
585049   235.682540  0.419340  0.720966  0.514710  0.000000  0.425830   
46719     10.913379  0.000000  0.000000  0.000000  0.000000  0.000000   
...             ...       ...       ...       ...       ...       ...   
1184477  172.431383  0.000000  0.000000  1.517910  0.035666  0.042216   
1039363   70.449342  0.823391  0.843409  0.000000  0.000000  0.000000   
1056114  193.747302  1.972430  0.900847  0.015660  0.839336  0.409984   
61335    232.524626  0.260011  0.807941  1.139190  0.000000  0.000000   
1106265  260.295692  0.000000  0.000000  0.047454  2.295650  0.070573   

         chroma_5  chroma_6  chroma_7  chroma_8  ...  chroma_14  chroma_15  \
409405   0.572059  1.198390  0.691098  0.0000

In [27]:
def paddington(data,padVal, div=100):
    
    reminder = len(data)%div

    if reminder == 0:
        return data

    amount_to_pad = div-reminder

    if data.ndim == 1:
        padded = np.pad(data,(0,amount_to_pad),constant_values=padVal)
    else:
        padded = np.pad(data, ((0, amount_to_pad), (0, 0)), constant_values=padVal) 
    return padded       

In [28]:
x_test_pad = paddington(x_test,padding)
x_train_pad = paddington(x_train,padding)
y_key_test = paddington(y_key_test,padding)
y_key_train = paddington(y_key_train,padding)
y_quality_test = paddington(y_quality_test,padding)
y_quality_train = paddington(y_quality_train,padding)

x_train_batches = x_train_pad.reshape(-1, 100, 25)
x_test_batches  = x_test_pad.reshape(-1, 100, 25)
y_key_train_batches = y_key_train.reshape(-1,100)
y_quality_train_batches = y_quality_train.reshape(-1,100)


In [32]:
inputs = Input(shape=(100,25))
output = layers.Conv1D(filters=24, kernel_size=3,strides=1,padding="same",activation='relu')(inputs)
output = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(output)

keyInput= layers.Dense(32,activation='relu')(output)
key= layers.Dense(14,activation= 'softmax',name='key')(keyInput)

shapeInput = layers.Dense(32,activation='relu')(output)
quality = layers.Dense(4,activation='softmax',name='quality')(shapeInput)

model = models.Model(inputs=inputs, outputs = [key, quality])
model.compile(optimizer='adam',
              loss= 'sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [36]:
model.fit(x_train_batches, {'key': y_key_train_batches, 'quality': y_quality_train_batches}, 
                      batch_size=32, 
                      epochs=5)



Epoch 1/5
355/355 [==============================] - 63s 178ms/step - loss: 1.9413 - key_loss: 1.2750 - quality_loss: 0.6664 - key_accuracy: 0.6223 - quality_accuracy: 0.7349
Epoch 2/5
355/355 [==============================] - 70s 196ms/step - loss: 1.9350 - key_loss: 1.2708 - quality_loss: 0.6642 - key_accuracy: 0.6232 - quality_accuracy: 0.7356
Epoch 3/5
355/355 [==============================] - 69s 194ms/step - loss: 1.9276 - key_loss: 1.2664 - quality_loss: 0.6613 - key_accuracy: 0.6240 - quality_accuracy: 0.7369
Epoch 4/5
355/355 [==============================] - 65s 182ms/step - loss: 1.9218 - key_loss: 1.2626 - quality_loss: 0.6591 - key_accuracy: 0.6248 - quality_accuracy: 0.7376
Epoch 5/5
355/355 [==============================] - 69s 194ms/step - loss: 1.9155 - key_loss: 1.2589 - quality_loss: 0.6567 - key_accuracy: 0.6255 - quality_accuracy: 0.7384


In [ ]:
model.save(filepath=REPLACE!!)

INFO:tensorflow:Assets written to: /home/viktor/ai2/toyModel/assets


INFO:tensorflow:Assets written to: /home/viktor/ai2/toyModel/assets


In [ ]:
df_to_pred = pd.read_csv(filepath_or_buffer=REPLACE!!,header=None)

df_to_pred = df_to_pred.drop(columns=[0])
df_to_pred.to_numpy()
padded_pred = paddington(df_to_pred,padding)
padded_pred = padded_pred.reshape(-1,100,25)
predictions = model.predict(padded_pred)

key_probs = predictions[0].reshape(-1, predictions[0].shape[-1])
quality_probs = predictions[1].reshape(-1, predictions[1].shape[-1])

keys_final = np.argmax(key_probs,axis=-1)
quality_final = np.argmax(quality_probs,axis= -1)

print(keys_final)

2/2 [==============================] - 1s 70ms/step
[12 12 12 ...  0  0  0]
